# Soluciones — Clase 29: Introducción a redes neuronales

Soluciones completas para todos los ejercicios de la clase.

> **Nota para el docente:** Compartir después de que los estudiantes hayan intentado resolver los ejercicios.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.neural_network import MLPClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, classification_report
import time

plt.style.use('seaborn-v0_8')
np.random.seed(42)

# Datos base
n = 400
cursos = np.random.choice(['A', 'B', 'C'], size=n, p=[0.4, 0.4, 0.2])
nota_mate = np.where(cursos == 'A', np.random.normal(7.5, 1.2, n),
    np.where(cursos == 'B', np.random.normal(7.0, 1.3, n), np.random.normal(5.8, 1.5, n))).clip(1, 10)
nota_lengua = np.where(cursos == 'A', np.random.normal(7.2, 1.1, n),
    np.where(cursos == 'B', np.random.normal(6.8, 1.2, n), np.random.normal(5.5, 1.4, n))).clip(1, 10)
asistencia = np.where(cursos == 'C', np.random.uniform(0.55, 0.85, n), np.random.uniform(0.70, 0.99, n))
edades = np.random.randint(18, 35, size=n)
prob = (nota_mate * 0.4 + nota_lengua * 0.4 + asistencia * 10 * 0.2) / 10
aprobado = (np.random.uniform(0, 1, n) < prob).astype(int)

df = pd.DataFrame({'nota_matematicas': nota_mate.round(1), 'nota_lengua': nota_lengua.round(1),
                   'asistencia': asistencia.round(2), 'edad': edades, 'aprobado': aprobado})

features = ['nota_matematicas', 'nota_lengua', 'asistencia', 'edad']
X = df[features]
y = df['aprobado']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc = scaler.transform(X_test)
print('Datos preparados')

## Solución Ejercicio 1: Cálculo manual de una neurona

In [ ]:
x = np.array([2.0, -1.0, 0.5])
w = np.array([0.3, 0.8, -0.5])
b = 0.1

z = np.dot(x, w) + b
print(f'z = {x[0]}×{w[0]} + {x[1]}×{w[1]} + {x[2]}×{w[2]} + {b}')
print(f'z = {x[0]*w[0]:.2f} + ({x[1]*w[1]:.2f}) + ({x[2]*w[2]:.2f}) + {b}')
print(f'z = {z:.4f}')

sigmoid = 1 / (1 + np.exp(-z))
relu = max(0, z)
print(f'\nSigmoid({z:.4f}) = {sigmoid:.4f}')
print(f'ReLU({z:.4f}) = {relu:.4f}')

# Con w1 = 1.0
w_nuevo = np.array([1.0, 0.8, -0.5])
z_nuevo = np.dot(x, w_nuevo) + b
sigmoid_nuevo = 1 / (1 + np.exp(-z_nuevo))
print(f'\nCon w1=1.0: z={z_nuevo:.4f}, sigmoid={sigmoid_nuevo:.4f}')
print(f'La salida aumentó de {sigmoid:.4f} a {sigmoid_nuevo:.4f}')
print('Un peso mayor en x1=2.0 (positivo) aumenta la salida de la neurona.')

## Solución Ejercicio 3 y 4: Comparación de arquitecturas

In [ ]:
configuraciones = [
    {'hidden_layer_sizes': (10,), 'activation': 'relu', 'max_iter': 200},
    {'hidden_layer_sizes': (100,), 'activation': 'relu', 'max_iter': 200},
    {'hidden_layer_sizes': (100, 50), 'activation': 'relu', 'max_iter': 300},
    {'hidden_layer_sizes': (100, 50), 'activation': 'tanh', 'max_iter': 300},
    {'hidden_layer_sizes': (200, 100, 50), 'activation': 'relu', 'max_iter': 500},
]

print(f'{"Capas":<25} {"Activación":<12} {"Precisión":>10} {"Tiempo":>8}')
print('-' * 58)

mejor_acc = 0
mejor_config = None
mejor_modelo = None

for config in configuraciones:
    inicio = time.time()
    modelo = MLPClassifier(**config, random_state=42)
    modelo.fit(X_train_sc, y_train)
    tiempo = time.time() - inicio
    acc = accuracy_score(y_test, modelo.predict(X_test_sc))
    print(f'{str(config["hidden_layer_sizes"]):<25} {config["activation"]:<12} {acc:>10.2%} {tiempo:>7.2f}s')
    if acc > mejor_acc:
        mejor_acc = acc
        mejor_config = config
        mejor_modelo = modelo

print(f'\nMejor configuración: {mejor_config["hidden_layer_sizes"]} con {mejor_config["activation"]}')
print(f'Precisión: {mejor_acc:.2%}')

## Solución Ejercicio 5: MLP vs Decision Tree vs Random Forest

In [ ]:
modelos_comparados = [
    ('MLP (mejor)', mejor_modelo, X_test_sc, True),
    ('Decision Tree', DecisionTreeClassifier(max_depth=5, random_state=42), X_test, False),
    ('Random Forest', RandomForestClassifier(n_estimators=100, random_state=42), X_test, False),
    ('Logistic Regression', LogisticRegression(random_state=42), X_test_sc, True),
]

print(f'{"Modelo":<25} {"Precisión":>10} {"Necesita escalar":>18}')
print('-' * 55)

accs = []
nombres = []

for nombre, modelo, X_ev, necesita_escalar in modelos_comparados:
    if nombre != 'MLP (mejor)' and nombre != 'Logistic Regression':
        modelo.fit(X_train, y_train)
    elif nombre == 'Logistic Regression':
        modelo.fit(X_train_sc, y_train)
    acc = accuracy_score(y_test, modelo.predict(X_ev))
    accs.append(acc)
    nombres.append(nombre)
    print(f'{nombre:<25} {acc:>10.2%} {"Sí" if necesita_escalar else "No":>18}')

# Visualizar comparación
plt.figure(figsize=(9, 4))
colores = ['#4C72B0', '#55A868', '#C44E52', '#8172B2']
bars = plt.bar(nombres, accs, color=colores)
plt.ylim(0.5, 1.0)
plt.title('Comparación de modelos — Precisión en test set')
plt.ylabel('Precisión')
for bar, acc in zip(bars, accs):
    plt.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.005,
             f'{acc:.1%}', ha='center', fontsize=10)
plt.tight_layout()
plt.show()

print()
print('CONCLUSIÓN: En datos tabulares con pocas filas, Random Forest es competitivo')
print('o superior al MLP, y no requiere normalización ni tanta configuración.')

## Solución Ejercicio 7: Curva de pérdida

In [ ]:
plt.figure(figsize=(10, 4))
plt.plot(mejor_modelo.loss_curve_, color='#4C72B0', linewidth=2)
plt.fill_between(range(len(mejor_modelo.loss_curve_)),
                 mejor_modelo.loss_curve_, alpha=0.1, color='#4C72B0')
plt.title('Curva de aprendizaje del mejor MLP')
plt.xlabel('Epoch')
plt.ylabel('Pérdida (loss)')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f'Pérdida inicial: {mejor_modelo.loss_curve_[0]:.4f}')
print(f'Pérdida final:   {mejor_modelo.loss_curve_[-1]:.4f}')
print(f'Reducción total: {(1 - mejor_modelo.loss_curve_[-1]/mejor_modelo.loss_curve_[0]):.1%}')
print()
print('La curva muestra que:')
print('1. El modelo aprende más rápido al principio (pendiente pronunciada)')
print('2. El aprendizaje se vuelve más lento a medida que se acerca al mínimo')
print('3. Al final la curva se estabiliza — el modelo convergió')